# 02. Ознаки

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.append(str(Path("..").resolve()))
from fabula_features import FEATURE_GROUPS, FEATURE_NAMES, FeatureExtractor

PROCESSED_DIR = Path("../data/processed")
WORDLIST_DIR = Path("../data/raw/cefr-j")
SPLITS = ["train", "validation", "test"]
LEVELS = ["A1", "A2", "B1", "B2", "C"]

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Групи ознак

In [2]:
pd.DataFrame(
    [(group, name) for group, names in FEATURE_GROUPS.items() for name in names],
    columns=["група", "ознака"],
).groupby("група", sort=False)["ознака"].apply(", ".join).to_frame()

,ознака
група,
length,"n_words, chars_per_word, n_syllables, syllable..."
lexical,"share_a1, share_a2, share_b1, share_b2, share_..."
grammar,"tree_depth, n_subordinate_clauses, n_finite_ve..."
dialogue,"n_contractions, n_interjections, is_question, ..."


## 2. Словник CEFR-J

In [3]:
extractor = FeatureExtractor(WORDLIST_DIR)
lexicon_levels = pd.Series(extractor.lexicon.by_word).map(lambda level: ["A1", "A2", "B1", "B2", "C1", "C2"][level])
lexicon_levels.value_counts().sort_index().to_frame("слів")

,слів
A1,1070
A2,1230
B1,2120
B2,2455
C1,925
C2,890


## 3. Перевірка на прикладах

In [4]:
examples = [
    "Hi Tom, how are you?",
    "Not really.",
    "Mom, I'm going to visit Grandma tomorrow!",
    "If I had known about the meeting, I would have come earlier.",
    "The treaty was ratified by the parliament despite considerable opposition.",
]
pd.DataFrame(extractor.extract_many(examples), index=examples).T.round(2)

,"Hi Tom, how are you?",Not really.,"Mom, I'm going to visit Grandma tomorrow!","If I had known about the meeting, I would have come earlier.",The treaty was ratified by the parliament despite considerable opposition.
n_words,5.00,2.00,8.00,12.00,10.00
chars_per_word,2.80,4.50,4.43,3.92,6.40
n_syllables,5.00,4.00,12.00,15.00,21.00
syllables_per_word,1.00,2.00,1.71,1.25,2.10
share_a1,1.00,1.00,1.00,1.00,0.40
share_a2,0.00,0.00,0.00,0.00,0.00
share_b1,0.00,0.00,0.00,0.00,0.30
share_b2,0.00,0.00,0.00,0.00,0.20
share_c,0.00,0.00,0.00,0.00,0.00
share_oov,0.00,0.00,0.00,0.00,0.10


## 4. Витягування

In [5]:
tables = {}
for split in SPLITS:
    sentences = pd.read_csv(PROCESSED_DIR / f"cefr_sp_{split}.csv")
    started = time.perf_counter()
    features = pd.DataFrame(extractor.extract_many(sentences["text"].tolist()), columns=FEATURE_NAMES)
    table = pd.concat([sentences[["text", "level", "level_id", "source"]], features], axis=1)
    table.to_csv(PROCESSED_DIR / f"features_{split}.csv", index=False, encoding="utf-8")
    tables[split] = table
    print(f"{split}: {len(table)} речень, {time.perf_counter() - started:.1f} с")

train: 7002 речень, 11.1 с


validation: 1500 речень, 2.0 с


test: 1501 речень, 2.0 с


## 5. Пропуски і константи

In [6]:
train = tables["train"]
summary = pd.DataFrame(
    {
        "пропусків": train[FEATURE_NAMES].isna().sum(),
        "унікальних значень": train[FEATURE_NAMES].nunique(),
        "частка ненульових": (train[FEATURE_NAMES] != 0).mean().round(3),
    }
)
summary.sort_values("частка ненульових")

,пропусків,унікальних значень,частка ненульових
has_vocative,0,2,0.001
is_exclamation,0,2,0.002
n_interjections,0,2,0.003
has_future,0,2,0.016
has_progressive,0,2,0.019
is_fragment,0,2,0.020
is_question,0,2,0.022
n_contractions,0,3,0.070
has_perfect,0,2,0.082
share_c,0,43,0.092


## 6. Середні значення за рівнями (train)

In [7]:
train.groupby("level")[FEATURE_NAMES].mean().reindex(LEVELS).T.round(2)

level,A1,A2,B1,B2,C
n_words,7.10,7.85,11.67,15.47,16.63
chars_per_word,3.82,4.09,4.58,4.96,5.37
n_syllables,7.31,9.29,15.79,23.23,27.47
syllables_per_word,1.21,1.26,1.41,1.56,1.70
share_a1,0.86,0.83,0.71,0.61,0.54
share_a2,0.11,0.10,0.13,0.12,0.11
share_b1,0.02,0.04,0.09,0.12,0.12
share_b2,0.01,0.01,0.03,0.05,0.05
share_c,0.00,0.00,0.00,0.01,0.02
share_oov,0.01,0.02,0.04,0.09,0.17


## 7. Висновки

- 33 ознаки в 4 групах: довжинні 4, лексичні 12, граматичні 10, діалогові 7.
- Пропусків немає; константних ознак немає.
- Монотонно зростають з рівнем: n_words, syllables_per_word, share_oov, max_word_level, tree_depth, n_subordinate_clauses, n_passive.
- Монотонно спадають: share_a1, min_zipf, mean_zipf, is_question, share_personal_pronouns.
- has_past спадає, has_present зростає — імовірно, ефект джерела (SCoRE — розповідні приклади, Wiki-Auto — енциклопедичний теперішній час).
- Діалогові ознаки в CEFR-SP майже нульові: звертання 0.1%, оклики 0.2%, вигуки 0.3%, неповні речення 2%.